In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

print(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE")
)

neo4j://127.0.0.1:7687 neo4j 12345678 movie-info


In [2]:
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

graph = Neo4jGraph()

# Text-to-Cypher

- Text-to-Cypher는 자연어 질의를 Neo4j 데이터베이스 쿼리 언어인 Cypher로 변환하여 지식 그래프를 효과적으로 검색할 수 있는 기술을 말한다.


## 자연어를 cypher query로 변환

In [3]:
################
#  직접 구현
################

graph.refresh_schema()
schema = graph.schema  # str
print(schema)
vector_index = graph.query("SHOW VECTOR INDEXES")
print(vector_index)

Node properties:
Movie {id: STRING, title: STRING, rating: FLOAT, released: LOCAL_DATE_TIME, overview: STRING, runtime: INTEGER, tagline: STRING}
Person {id: STRING, name: STRING}
Genre {id: STRING, name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
[]


In [4]:
model = ChatOpenAI(model="gpt-5.4-mini")
prompt_template = PromptTemplate(
    template="""
<instruction>
다음 그래프 스키마를 참고해서 질문에 맞는 cypher 쿼리를 작성하세요.
</instruction>

<input_data>
    <schema>{schema}</schema>
    <question>{query}</question>
</input_data>

<constraint>
- 질문과 schema 가 맞지 않으면 vector-index를 참고해서 vector 검색쿼리를 생성한다.
- 생성한 쿼리만 문자열로 반환한다. 설명과 같은 다른 어떠한 것도 추가로 작성하지 않는다.
- 만약 감독의 이름, 영화제목, 배우이름이 쿼리문에 들어갈 경우 영어로 바꿔서 입력한다.
</constraint>
"""
)

query_chain = prompt_template | model | StrOutputParser()

def text_to_query(question: str, schema: str) -> str:
    """자연어 질의를 cypher query로 변환하는 함수"""
    return query_chain.invoke({"schema":schema, "query":question})


In [5]:
query = "톰 행크스가 출연한 영화는?"
# query = "톰 행크스가 출연한 영화들의 감독들은 누구인가요?"
# query = "2차 세계대전을 배경으로 치열한 전투를 보여주는 영화의 제목과 그 영화 감독, 배우를 알려줘."

cypher_query = text_to_query(query, schema)
print(cypher_query)

print("생성된 쿼리로 조회")
resp = graph.query(cypher_query)
print("-------------조회결과--------------------")
pprint(resp)

MATCH (p:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)
RETURN m.title
생성된 쿼리로 조회
-------------조회결과--------------------
[{'m.title': 'Splash'},
 {'m.title': 'Big'},
 {'m.title': 'A League of Their Own'},
 {'m.title': 'Philadelphia'},
 {'m.title': 'Forrest Gump'},
 {'m.title': 'Apollo 13'},
 {'m.title': 'Toy Story'},
 {'m.title': 'That Thing You Do!'},
 {'m.title': 'Saving Private Ryan'},
 {'m.title': "You've Got Mail"},
 {'m.title': 'Toy Story 2'},
 {'m.title': 'The Green Mile'},
 {'m.title': 'Cast Away'},
 {'m.title': 'Road to Perdition'},
 {'m.title': 'Catch Me If You Can'},
 {'m.title': 'The Ladykillers'},
 {'m.title': 'The Terminal'},
 {'m.title': 'The Polar Express'},
 {'m.title': 'The Da Vinci Code'},
 {'m.title': 'Who Killed the Electric Car?'},
 {'m.title': "Charlie Wilson's War"},
 {'m.title': 'Angels & Demons'},
 {'m.title': 'Toy Story 3'},
 {'m.title': 'Larry Crowne'},
 {'m.title': 'Extremely Loud & Incredibly Close'},
 {'m.title': 'Cloud Atlas'},
 {'m.title': 'Captain P

# `GraphCypherQAChain` 클래스 사용

- `from langchain_neo4j import GraphCypherQAChain`

## 기능

`GraphCypherQAChain`은 자연어로 Neo4j 데이터베이스와 대화할 수 있도록 하는 체인이다. LLM과 DB 스키마를 활용해 사용자 질문을 Cypher 쿼리로 변환하고 이를 DB에 실행한다.

내부적으로 자연어를 cypher 쿼리로 만드는 것과 질문과 조회결과를 바탕으로 최종 답변을 생성할 것 두 단계로 실행된다.

```
사용자 자연어 질문
    ↓ [1단계 LLM] 그래프 스키마 참고
Cypher 쿼리 생성
    ↓ Neo4j 실행
쿼리 결과 (컨텍스트)
    ↓ [2단계 LLM] 컨텍스트 + 원래 질문
최종 자연어 답변
```


## 객체 생성 — `from_llm()` 클래스 메소드 사용

- 객체 initializer 대신 **`from_llm()` 클래스 메소드**를 사용하여 생성한다.

    ```python
    chain = GraphCypherQAChain.from_llm(
        llm=None,
        *,
        graph,                        # 필수
        allow_dangerous_requests,     # 필수 (True 명시 필요)
        qa_prompt=None,
        cypher_prompt=None,
        cypher_llm=None,
        qa_llm=None,
        exclude_types=[],
        include_types=[],
        validate_cypher=False,
        qa_llm_kwargs=None,
        cypher_llm_kwargs=None,
        use_function_response=False,
        function_response_system=...,
        top_k=10,
        return_intermediate_steps=False,
        return_direct=False,
        verbose=False,
    )
    ```

### 주요 파라미터 정리

**필수 파라미터**

| 파라미터 | 타입 | 설명 |
|---|---|---|
| `graph` | `Neo4jGraph` | 연결할 Neo4j 그래프 객체 |
| `allow_dangerous_requests` | `bool` | `True` 필수. DB 변경 쿼리 허용에 대한 사용자 동의 명시 |

**LLM 설정**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `llm` | `None` | Cypher 생성 + 답변 생성에 동일한 LLM 사용 |
| `cypher_llm` | `None` | Cypher 생성 전용 LLM (llm과 중복 사용 불가) |
| `qa_llm` | `None` | 답변 생성 전용 LLM (llm과 중복 사용 불가) |

**프롬프트 커스터마이징**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `cypher_prompt` | (CYPHER_GENERATION_PROMPT) | Cypher 생성용 커스텀 프롬프트 (`{schema}`, `{question}` 변수 포함). 생략시 내장된 프롬프트 사용 |
| `qa_prompt` | (CYPHER_QA_PROMPT) | 답변 생성용 커스텀 프롬프트. 생략시 내장된 프롬프트 사용 |
| `function_response_system` | (FUNCTION_RESPONSE_SYSTEM) | `use_function_response=True` 일 때 시스템 메시지 |

**스키마 필터링**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `include_types` | `[]` | LLM에 노출할 노드/관계 타입만 지정 |
| `exclude_types` | `[]` | LLM에서 제외할 노드/관계 타입 지정 |

**결과 제어**

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `top_k` | `10` | 쿼리 결과를 LLM에 전달할 최대 건수 |
| `return_intermediate_steps` | `False` | `True`이면 생성된 Cypher, 쿼리 결과도 함께 반환 |
| `return_direct` | `False` | `True`이면 LLM 2단계 생략, DB 조회 결과를 직접 반환 |
| `validate_cypher` | `False` | `True`이면 관계 방향 오류 등을 자동 교정 |
| `use_function_response` | `False` | `True`이면 DB 결과를 tool/function 응답 형식으로 LLM에 전달 (정확도 향상) |
| `verbose` | `False` | `True`이면 생성된 Cypher, 컨텍스트 등 중간 과정 출력 |


## 주요 메소드, 속성

- `invoke()` — 질의응답 실행
  - 기본 반환 값 구조
     ```python
     result = chain.invoke({"query": "Top Gun에 출연한 배우는?"})
     print(result["result"])   # 최종 자연어 답변
     ```

  - `return_intermediate_steps=True` 시 반환 값 구조:

     ```python
     {
        "query": "Top Gun에 출연한 배우는?",
        "result": "Tom Cruise, Val Kilmer ...",
        "intermediate_steps": [
           {"query": "MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.name='Top Gun' RETURN a.name"},
           {"context": [{"a.name": "Tom Cruise"}, {"a.name": "Val Kilmer"}, ...]}
        ]
     }
     ```
### 주요 속성

| 속성 | 설명 |
|---|---|
| `graph_schema` | LLM에 전달되는 현재 그래프 스키마 문자열 |
| `input_key` | 입력 키 (기본값: `"query"`) |
| `output_key` | 출력 키 (기본값: `"result"`) |

- Cypher/QA LLM 분리 + 커스텀 프롬프트 하기

    ```python
    from langchain_core.prompts import PromptTemplate

    CYPHER_TEMPLATE = """스키마: {schema}
    질문을 Cypher로만 변환해줘. 설명 없이 쿼리만 출력.
    질문: {question}"""

    chain = GraphCypherQAChain.from_llm(
        graph=graph,
        cypher_llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),  # Cypher 생성 전용
        qa_llm=ChatOpenAI(model="gpt-4o", temperature=0),           # 답변 생성 전용
        cypher_prompt=PromptTemplate(
            input_variables=["schema", "question"],
            template=CYPHER_TEMPLATE
        ),
        top_k=5,
        return_intermediate_steps=True,
        validate_cypher=True,
        allow_dangerous_requests=True,
    )
    ```

In [6]:
# GraphCypherQAChain 이 사용하는 프롬프트 조회
from langchain_neo4j.chains.graph_qa.cypher import (
    CYPHER_GENERATION_PROMPT,   # 기본 Cypher 생성 프롬프트: 자연어 -> cypher 쿼리 생성 프롬프트
    CYPHER_QA_PROMPT,           # 기본 QA 프롬프트 : cypher 결과 -> 자연어로 변환시 사용할 프롬프트
)
from langchain_core.prompts import PromptTemplate

print(type(CYPHER_GENERATION_PROMPT), type(CYPHER_QA_PROMPT))
print(CYPHER_GENERATION_PROMPT.template)
print("="*50)
print(CYPHER_QA_PROMPT.template)

################################################
# 사용자 정의 프롬프트 생성 
# - 기존 프롬프트를 수정하는 방식으로 만든다.
################################################
custom_prompt_str = CYPHER_GENERATION_PROMPT.template+\
"""
<constraint>
- When filtering by date, convert the data to the `date` type for processing.
</constraint>
"""

custom_cypher_template = PromptTemplate(template=custom_prompt_str)
print(custom_cypher_template.input_variables)


<class 'langchain_core.prompts.prompt.PromptTemplate'> <class 'langchain_core.prompts.prompt.PromptTemplate'>
Task:Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.
Schema:
{schema}
Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

Examples (optional):
{examples}

The question is:
{question}
You are an assistant that helps to form nice and human understandable answers.
The information part contains the provided information that you must use to construct an answer.
The provided information is authoritative, you must never doubt it or try to use your internal knowledge to correct it.
Make the answer sound as a response to the question. Do

In [ ]:
#################################
# CypherChain생성
#################################
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm, 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=custom_cypher_template # PromptTemplate (Cypher 쿼리 생성시 사용.)
)



In [10]:
# print(cypher_chain.graph_schema)
print("--------------입/출력 key-------------------")
print(cypher_chain.input_key,"/", cypher_chain.output_key)


--------------입/출력 key-------------------
query / result


## 쿼리 실행

In [11]:
answer = cypher_chain.invoke({"query": "영화 'Apollo 13'에 대한 정보를 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Apollo 13'})
OPTIONAL MATCH (m)-[:IN_GENRE]->(g:Genre)
OPTIONAL MATCH (p:Person)-[:ACTED_IN]->(m)
OPTIONAL MATCH (d:Person)-[:DIRECTED]->(m)
RETURN m, collect(DISTINCT g) AS genres, collect(DISTINCT p) AS actors, collect(DISTINCT d) AS directors
Full Context:
[{'m': {'overview': 'The true story of technical troubles that scuttle the Apollo 13 lunar mission in 1971, risking the lives of astronaut Jim Lovell and his crew, with the failed journey turning into a thrilling saga of heroism. Drifting more than 200,000 miles from Earth, the astronauts work furiously with the ground crew to avert tragedy.', 'rating': 7.3, 'runtime': 140, 'tagline': 'Houston, we have a problem.', 'id': 'movie-635', 'title': 'Apollo 13', 'released': neo4j.time.DateTime(1995, 6, 30, 0, 0, 0, 0)}, 'genres': [{'name': 'Drama', 'id': 'genre-Drama'}], 'actors': [{'name': 'Kevin Bacon', 'id': 'person-Kevin Bacon'}, {'name': 'Ed Harri

In [12]:
print(answer)

{'query': "영화 'Apollo 13'에 대한 정보를 알려주세요.", 'result': '영화 Apollo 13은 1971년 아폴로 13호 달 임무의 기술적 문제로 인해 우주비행사 짐 러벨과 승무원들의 생존이 위협받는 실화를 다룬 드라마 영화입니다. 지구에서 20만 마일 이상 떨어진 곳에서 우주비행사들과 지상팀이 함께 재난을 막기 위해 분투하는 이야기를 담고 있습니다.  \n\n- 평점: 7.3\n- 상영 시간: 140분\n- 태그라인: “Houston, we have a problem.”\n- 개봉일: 1995년 6월 30일\n- 장르: 드라마\n- 감독: Ron Howard\n- 출연: Kevin Bacon, Ed Harris, Tom Hanks, Bill Paxton, Gary Sinise'}


In [13]:
from pprint import pprint
pprint(answer['result'])

('영화 Apollo 13은 1971년 아폴로 13호 달 임무의 기술적 문제로 인해 우주비행사 짐 러벨과 승무원들의 생존이 위협받는 실화를 '
 '다룬 드라마 영화입니다. 지구에서 20만 마일 이상 떨어진 곳에서 우주비행사들과 지상팀이 함께 재난을 막기 위해 분투하는 이야기를 담고 '
 '있습니다.  \n'
 '\n'
 '- 평점: 7.3\n'
 '- 상영 시간: 140분\n'
 '- 태그라인: “Houston, we have a problem.”\n'
 '- 개봉일: 1995년 6월 30일\n'
 '- 장르: 드라마\n'
 '- 감독: Ron Howard\n'
 '- 출연: Kevin Bacon, Ed Harris, Tom Hanks, Bill Paxton, Gary Sinise')


In [14]:
answer = cypher_chain.invoke({"query":"Apollo 13에 출연한 배우이름과 그들이 출연한 다른 영화들을 조회해 주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Apollo 13'})
MATCH (p)-[:ACTED_IN]->(other:Movie)
WHERE other.title <> 'Apollo 13'
RETURN p.name AS actorName, collect(DISTINCT other.title) AS otherMovies
ORDER BY actorName
Full Context:
[{'actorName': 'Bill Paxton', 'otherMovies': ['Tombstone', 'True Lies', 'Twister', 'A Simple Plan', 'Mighty Joe Young', 'U-571', 'Vertical Limit', 'Frailty', 'Haywire', 'Shanghai Calling', '2 Guns', 'Million Dollar Arm', 'Edge of Tomorrow', 'Nightcrawler']}, {'actorName': 'Ed Harris', 'otherMovies': ['The Right Stuff', 'The Abyss', 'Glengarry Glen Ross', 'The Firm', 'Nixon', 'Eye for an Eye', 'The Rock', 'Absolute Power', 'The Truman Show', 'Stepmom', 'Pollock', 'Enemy at the Gates', 'Buffalo Soldiers', 'A Beautiful Mind', 'Radio', 'Winter Passing', 'Copying Beethoven', 'Appaloosa', 'Salvation Boulevard', 'Man on a Ledge', 'Pain & Gain', 'Snowpiercer', 'Gravity', 'Run All Night']}, {'actorNa

In [15]:
print(answer["result"])

Bill Paxton: Tombstone, True Lies, Twister, A Simple Plan, Mighty Joe Young, U-571, Vertical Limit, Frailty, Haywire, Shanghai Calling, 2 Guns, Million Dollar Arm, Edge of Tomorrow, Nightcrawler  
Ed Harris: The Right Stuff, The Abyss, Glengarry Glen Ross, The Firm, Nixon, Eye for an Eye, The Rock, Absolute Power, The Truman Show, Stepmom, Pollock, Enemy at the Gates, Buffalo Soldiers, A Beautiful Mind, Radio, Winter Passing, Copying Beethoven, Appaloosa, Salvation Boulevard, Man on a Ledge, Pain & Gain, Snowpiercer, Gravity, Run All Night  
Gary Sinise: Forrest Gump, Albino Alligator, Ransom, Snake Eyes, Reindeer Games, Mission to Mars, Impostor, The Big Bounce, Open Season  
Kevin Bacon: Diner, Footloose, Tremors, Flatliners, JFK, A Few Good Men, The River Wild, Sleepers, Wild Things, Stir of Echoes, My Dog Skip, Hollow Man, Mystic River, Where the Truth Lies, Death Sentence, Frost/Nixon, Super, X-Men: First Class, R.I.P.D., Black Mass  
Tom Hanks: Splash, Big, A League of Their Own,

In [16]:
answer = cypher_chain.invoke({"query": "'Christopher Nolan' 감독의 영화를 모두 찾아주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie)
RETURN m.title AS title, m.released AS released, m.rating AS rating, m.overview AS overview, m.runtime AS runtime, m.tagline AS tagline, m.id AS id
Full Context:
[{'title': 'Memento', 'released': neo4j.time.DateTime(2000, 10, 11, 0, 0, 0, 0), 'rating': 8.1, 'overview': "Suffering short-term memory loss after a head injury, Leonard Shelby embarks on a grim quest to find the lowlife who murdered his wife in this gritty, complex thriller that packs more knots than a hangman's noose. To carry out his plan, Shelby snaps Polaroids of people and places, jotting down contextual notes on the backs of photos to aid in his search and jog his memory. He even tattoos his own body in a desperate bid to remember.", 'runtime': 113, 'tagline': 'Some memories are best forgotten.', 'id': 'movie-3573'}, {'title': 'Insomnia', 'released': neo4j.time.DateTime(2002, 5, 24, 0, 0, 0

In [17]:
print(answer)

{'query': "'Christopher Nolan' 감독의 영화를 모두 찾아주세요.", 'result': 'Memento, Insomnia, Batman Begins, The Prestige, The Dark Knight, Inception, The Dark Knight Rises, Interstellar입니다.'}


In [18]:
pprint(answer['result'])

('Memento, Insomnia, Batman Begins, The Prestige, The Dark Knight, Inception, '
 'The Dark Knight Rises, Interstellar입니다.')


In [19]:
answer = cypher_chain.invoke({"query": "'Tom Hanks'가 출연한, 평점이 가장 높은 영화 3개는 무엇인가요? 영화제목과 평점을 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
WHERE p.name = 'Tom Hanks'
RETURN m.title AS title, m.rating AS rating
ORDER BY m.rating DESC
LIMIT 3
Full Context:
[{'title': 'Forrest Gump', 'rating': 8.2}, {'title': 'The Green Mile', 'rating': 8.2}, {'title': 'Saving Private Ryan', 'rating': 7.9}]

> Finished chain.


In [20]:
print(answer)

{'query': "'Tom Hanks'가 출연한, 평점이 가장 높은 영화 3개는 무엇인가요? 영화제목과 평점을 알려주세요.", 'result': 'Tom Hanks가 출연한 영화 중 평점이 가장 높은 3개는 다음과 같습니다.\n\n1. **Forrest Gump** — **8.2**\n2. **The Green Mile** — **8.2**\n3. **Saving Private Ryan** — **7.9**'}


In [21]:
pprint(answer['result'])

('Tom Hanks가 출연한 영화 중 평점이 가장 높은 3개는 다음과 같습니다.\n'
 '\n'
 '1. **Forrest Gump** — **8.2**\n'
 '2. **The Green Mile** — **8.2**\n'
 '3. **Saving Private Ryan** — **7.9**')


In [22]:
answer = cypher_chain.invoke({"query": "2000년 이후 개봉한 'Drama' 장르 영화 중 평점이 높은 순서로 5개를 영화제목, 개봉일자, 평점항목을 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)-[:IN_GENRE]->(:Genre {name: 'Drama'})
WHERE date(m.released) > date('2000-01-01')
RETURN m.title AS 영화제목, m.released AS 개봉일자, m.rating AS 평점
ORDER BY m.rating DESC
LIMIT 5
Full Context:
[{'영화제목': 'Me You and Five Bucks', '개봉일자': neo4j.time.DateTime(2015, 7, 7, 0, 0, 0, 0), '평점': 10.0}, {'영화제목': 'Whiplash', '개봉일자': neo4j.time.DateTime(2014, 10, 10, 0, 0, 0, 0), '평점': 8.3}, {'영화제목': 'The Visual Bible: The Gospel of John', '개봉일자': neo4j.time.DateTime(2003, 9, 11, 0, 0, 0, 0), '평점': 8.2}, {'영화제목': 'The Dark Knight', '개봉일자': neo4j.time.DateTime(2008, 7, 16, 0, 0, 0, 0), '평점': 8.2}, {'영화제목': 'Guten Tag, Ramón', '개봉일자': neo4j.time.DateTime(2013, 10, 18, 0, 0, 0, 0), '평점': 8.1}]

> Finished chain.


In [23]:
print(answer)

{'query': "2000년 이후 개봉한 'Drama' 장르 영화 중 평점이 높은 순서로 5개를 영화제목, 개봉일자, 평점항목을 알려주세요.", 'result': 'Me You and Five Bucks, 2015-07-07T00:00:00, 10.0  \nWhiplash, 2014-10-10T00:00:00, 8.3  \nThe Visual Bible: The Gospel of John, 2003-09-11T00:00:00, 8.2  \nThe Dark Knight, 2008-07-16T00:00:00, 8.2  \nGuten Tag, Ramón, 2013-10-18T00:00:00, 8.1'}


In [24]:
pprint(answer['result'])

('Me You and Five Bucks, 2015-07-07T00:00:00, 10.0  \n'
 'Whiplash, 2014-10-10T00:00:00, 8.3  \n'
 'The Visual Bible: The Gospel of John, 2003-09-11T00:00:00, 8.2  \n'
 'The Dark Knight, 2008-07-16T00:00:00, 8.2  \n'
 'Guten Tag, Ramón, 2013-10-18T00:00:00, 8.1')


### 출력 갯수를 지정 (top k)

In [25]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    
    top_k=5,                         # 최대 5개의 결과만 반환
)

answer = cypher_chain.invoke({"query": "'Tom Hanks' 배우가 출연한 영화를 모두 알려주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)
RETURN m.title AS title, m.released AS released, m.rating AS rating, m.overview AS overview, m.runtime AS runtime, m.tagline AS tagline
Full Context:
[{'title': 'Splash', 'released': neo4j.time.DateTime(1984, 3, 9, 0, 0, 0, 0), 'rating': 6.1, 'overview': "A successful businessman falls in love with the girl of his dreams. There's one big complication though; he's fallen hook, line and sinker for a mermaid.", 'runtime': 111, 'tagline': "Allen Bauer Thought He'd Never Find The Right Woman... He Was Only Half Wrong!"}, {'title': 'Big', 'released': neo4j.time.DateTime(1988, 6, 3, 0, 0, 0, 0), 'rating': 6.9, 'overview': 'A young boy, Josh Baskin makes a wish at a carnival machine to be big. He wakes up the following morning to find that it has been granted and his body has grown older overnight. But he is still the same 13-year-old boy inside. Now he must learn how to c

In [26]:
# LLM 답변 출력
pprint(answer['result'])

'Splash, Big, Philadelphia, Forrest Gump에 Tom Hanks가 출연했습니다.'


### 생성된 cypher쿼리 포함 반환

- `return_intermediate_steps`=True

In [27]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_intermediate_steps=True  # 생성된 Cypher 쿼리와 중간 결과 확인
)

answer = cypher_chain.invoke({"query": "평점이 8점 이상인 'Action' 장르 영화는 무엇이 있나요?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
WHERE g.name = 'Action' AND m.rating >= 8
RETURN m.title AS title, m.rating AS rating
ORDER BY m.rating DESC, m.title ASC
Full Context:
[{'title': "One Man's Hero", 'rating': 9.3}, {'title': 'Seven Samurai', 'rating': 8.2}, {'title': 'The Dark Knight', 'rating': 8.2}, {'title': 'The Empire Strikes Back', 'rating': 8.2}, {'title': 'Inception', 'rating': 8.1}, {'title': 'Star Wars', 'rating': 8.1}, {'title': 'The Lord of the Rings: The Return of the King', 'rating': 8.1}, {'title': 'Oldboy', 'rating': 8.0}, {'title': 'Scarface', 'rating': 8.0}, {'title': 'Star Wars: Clone Wars: Volume 1', 'rating': 8.0}]

> Finished chain.


In [28]:
answer['intermediate_steps']

[{'query': "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE g.name = 'Action' AND m.rating >= 8\nRETURN m.title AS title, m.rating AS rating\nORDER BY m.rating DESC, m.title ASC"},
 {'context': [{'title': "One Man's Hero", 'rating': 9.3},
   {'title': 'Seven Samurai', 'rating': 8.2},
   {'title': 'The Dark Knight', 'rating': 8.2},
   {'title': 'The Empire Strikes Back', 'rating': 8.2},
   {'title': 'Inception', 'rating': 8.1},
   {'title': 'Star Wars', 'rating': 8.1},
   {'title': 'The Lord of the Rings: The Return of the King', 'rating': 8.1},
   {'title': 'Oldboy', 'rating': 8.0},
   {'title': 'Scarface', 'rating': 8.0},
   {'title': 'Star Wars: Clone Wars: Volume 1', 'rating': 8.0}]}]

In [29]:
for k, v in answer.items():
    print(f"{k}: {v}")

query: 평점이 8점 이상인 'Action' 장르 영화는 무엇이 있나요?
result: 평점이 8점 이상인 영화는 One Man's Hero, Seven Samurai, The Dark Knight, The Empire Strikes Back, Inception, Star Wars, The Lord of the Rings: The Return of the King, Oldboy, Scarface, Star Wars: Clone Wars: Volume 1 입니다.
intermediate_steps: [{'query': "MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE g.name = 'Action' AND m.rating >= 8\nRETURN m.title AS title, m.rating AS rating\nORDER BY m.rating DESC, m.title ASC"}, {'context': [{'title': "One Man's Hero", 'rating': 9.3}, {'title': 'Seven Samurai', 'rating': 8.2}, {'title': 'The Dark Knight', 'rating': 8.2}, {'title': 'The Empire Strikes Back', 'rating': 8.2}, {'title': 'Inception', 'rating': 8.1}, {'title': 'Star Wars', 'rating': 8.1}, {'title': 'The Lord of the Rings: The Return of the King', 'rating': 8.1}, {'title': 'Oldboy', 'rating': 8.0}, {'title': 'Scarface', 'rating': 8.0}, {'title': 'Star Wars: Clone Wars: Volume 1', 'rating': 8.0}]}]


### Neo4j DB 반환결과만 얻기

- `return_direct`=True
- cypher 쿼리 실행 결과를 바탕으로 LLM이 응답메세지를 만들어 반환하는데 `return_direct=True` 로 하면 cypher 쿼리 결과를 그대로 dictionary로 반환한다.

In [37]:
# 직접 Cypher 결과 얻기 (LLM 답변 없이)
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    return_direct=True,   # LLM 답변 생성 단계 건너뛰기
    cypher_prompt=custom_cypher_template
)

# answer = cypher_chain.invoke({"query": "2000년대(2000-01-01 ~ 2009-12-31) 개봉한 영화를 평점 순으로 정렬해주세요."})
answer = cypher_chain.invoke({"query": "톰 행크스와 레오나르도 디카프리오가 같이 출연한 영화를 찾아주세요."})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
WHERE p1.name = '톰 행크스' AND p2.name = '레오나르도 디카프리오'
RETURN DISTINCT m.title AS title, m.id AS id, m.released AS released, m.rating AS rating

> Finished chain.


In [38]:
for k, v in answer.items():
    print(f"{k}: {v}")

query: 톰 행크스와 레오나르도 디카프리오가 같이 출연한 영화를 찾아주세요.
result: []


In [39]:
import pandas as pd
pd.DataFrame([item for item in answer['result']])

""


### cypher 쿼리 생성 모델과 최종 답변 생성 모델을 다르게 적용

In [ ]:
cypher_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.0)
qa_llm = ChatOpenAI(model="gpt-5.5", temperature=0.0)

cypher_chain = GraphCypherQAChain.from_llm(
    cypher_llm=cypher_llm,  # Cypher 쿼리 생성용 LLM
    qa_llm=qa_llm,          # 최종 답변 생성용 LLM
    
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
)

answer = cypher_chain.invoke({"query": "배우 'Leonardo DiCaprio'와 감독 'Christopher Nolan'이 함께 작업한 영화가 있나요?"})

In [ ]:
for k, v in answer.items():
    print(f"{k}: {v}")